# Technology Conference IF - Engelhart / Ensimag - 23/09/2025

Welcome to this data science challenge on **algorithmic trading on electricity markets**!

**Instructions:**
- Run the first two code cells. This will create a **submission.csv** file with a very simple forecast (linear regression).
- Submit the forecast (on the right, Submit to competition => Submit), and check that your name appears in the [Leaderboard](https://www.kaggle.com/competitions/ensimag-if-2025/leaderboard).
- Modify the code and make other submissions to improve your score!
- (You can save data in /kaggle/working and save your notebook at the top right)

**Some ideas to consider:**
- Explore the data. Calculate statistics on the different variables and visualise the data by plotting curves (using matplotlib or plotly, for example)
- Your number of submissions is limited, so you can test your model first on the training set (train.csv) by performing cross-validation
- Is there any strange/corrupt data? (Is it relevant for the model to learn with this data?)
- Is linear regression really appropriate?
- Based on the existing data, can new variables be added? (Temporal variables? Variable transformation?)

### Importing the necessary libraries

In [19]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone

### Load and explore data

In [20]:
# Load data
train = pd.read_csv('/kaggle/input/ensimag-if-2025/train.csv', parse_dates=['date'], index_col='date')
test = pd.read_csv('/kaggle/input/ensimag-if-2025/test.csv', parse_dates=['date'])

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {train.columns.tolist()}")
print(f"\nFirst few rows:")
print(train.head())

print(f"\nSpread statistics:")
print(train['spread'].describe())

Train shape: (140157, 4)
Test shape: (24138, 5)

Train columns: ['wind', 'solar', 'load', 'spread']

First few rows:
                       wind  solar     load  spread
date                                               
2020-01-01 00:00:00  6084.0    0.0  43915.0   38.40
2020-01-01 00:15:00  5739.0    0.0  43770.0 -105.93
2020-01-01 00:30:00  5774.0    0.0  43267.0   -1.48
2020-01-01 00:45:00  5804.0    0.0  42934.0    3.97
2020-01-01 01:00:00  5791.0    0.0  42718.0  100.09

Spread statistics:
count    140157.000000
mean          0.889667
std         220.876249
min       -9271.590000
25%         -61.510000
50%           3.230000
75%          63.090000
max       15781.480000
Name: spread, dtype: float64


### Quick Visualisation

In [21]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train['spread'], mode='lines', name='spread'))
fig.update_layout(title='Spread Over Time', xaxis_title='Date', yaxis_title='Spread (€/MWh)')
fig.show()

### Feature Engineering

In [22]:
# --- ensure we have a real datetime column called 'date' in BOTH train and test ---

# Train
train = train.reset_index()
if 'date' not in train.columns:
    train = train.rename(columns={'index': 'date'})

train['date'] = pd.to_datetime(train['date'])

# Test
test = test.reset_index()
if 'date' not in test.columns:
    test = test.rename(columns={'index': 'date'})

test['date'] = pd.to_datetime(test['date'])

print("train columns:", train.columns.tolist()[:10])
print("test columns:", test.columns.tolist()[:10])
print("train date sample:", train['date'].head().tolist())

train columns: ['date', 'wind', 'solar', 'load', 'spread']
test columns: ['index', 'ID', 'date', 'wind', 'solar', 'load']
train date sample: [Timestamp('2020-01-01 00:00:00'), Timestamp('2020-01-01 00:15:00'), Timestamp('2020-01-01 00:30:00'), Timestamp('2020-01-01 00:45:00'), Timestamp('2020-01-01 01:00:00')]


In [23]:
def create_features(df):
    """Add features to the dataframe"""
    df = df.copy()

    # Use the actual 'date' column (NOT the index anymore)
    dt = df['date']

    # Temporal features
    df['hour'] = dt.dt.hour
    df['dayofweek'] = dt.dt.dayofweek
    df['month'] = dt.dt.month
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

    # Renewable features (KEY INSIGHT FROM PAPER)
    df['renewable_gen'] = df['wind'] + df['solar']
    df['renewable_pct'] = df['renewable_gen'] / df['load']
    df['renewable_pct2'] = df['renewable_pct'] ** 2
    df['renewable_pct3'] = df['renewable_pct'] ** 3

    # Net demand (very important for power markets)
    df['net_demand'] = df['load'] - df['renewable_gen']
    df['net_demand_ratio'] = df['net_demand'] / df['load']

    return df

def add_market_dynamics(df):
    """
    Features that capture short-term grid stress / ramps.
    These are critical for electricity price formation.
    """
    df = df.copy()

    # 15-minute ramps (most predictive features)
    df['wind_ramp'] = df['wind'].diff()
    df['solar_ramp'] = df['solar'].diff()
    df['load_ramp'] = df['load'].diff()

    # Net-demand ramp (very important)
    df['net_demand_ramp'] = df['net_demand'].diff()

    # Short-term volatility (system stress proxy)
    df['wind_vol_1h'] = df['wind'].rolling(4).std()
    df['load_vol_1h'] = df['load'].rolling(4).std()
    df['net_demand_vol_1h'] = df['net_demand'].rolling(4).std()

    # Lag structure (market memory)
    df['net_demand_lag_1h'] = df['net_demand'].shift(4)
    df['net_demand_lag_24h'] = df['net_demand'].shift(96)

    return df

# Apply feature engineering
train_fe = create_features(train)
test_fe = create_features(test)

train_fe = add_market_dynamics(train_fe)
test_fe = add_market_dynamics(test_fe)

print(f"New features created: {[c for c in train_fe.columns if c not in train.columns]}")

New features created: ['hour', 'dayofweek', 'month', 'is_weekend', 'renewable_gen', 'renewable_pct', 'renewable_pct2', 'renewable_pct3', 'net_demand', 'net_demand_ratio', 'wind_ramp', 'solar_ramp', 'load_ramp', 'net_demand_ramp', 'wind_vol_1h', 'load_vol_1h', 'net_demand_vol_1h', 'net_demand_lag_1h', 'net_demand_lag_24h']


In [24]:
# ---- REMOVE ROWS CREATED BY LAGS / ROLLING WINDOWS ----
train_fe = train_fe.replace([np.inf, -np.inf], np.nan)
train_fe = train_fe.dropna().reset_index(drop=True)

print("After cleaning:")
print("Shape:", train_fe.shape)
print("Remaining NaNs:", train_fe.isna().sum().sum())

After cleaning:
Shape: (140061, 24)
Remaining NaNs: 0


### Prepare Data

In [25]:
# Features to use
feature_cols = ['hour', 'dayofweek', 'month', 'is_weekend',
                'wind', 'solar', 'load',
                'renewable_gen', 'renewable_pct', 'renewable_pct2', 'renewable_pct3',
                'net_demand', 'net_demand_ratio']

x_train = train_fe[feature_cols].fillna(0)
y_train = train_fe['spread']

x_test = test_fe.set_index('ID')[feature_cols].fillna(0)

print(f"Training with {len(feature_cols)} features")
print(f"X_train shape: {x_train.shape}")
print(f"X_test shape: {x_test.shape}")

Training with 13 features
X_train shape: (140061, 13)
X_test shape: (24138, 13)


### Train Model

In [26]:
# Use Gradient Boosting instead of Linear Regression
model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(x_train, y_train)
print("Model trained!")

# Feature importance
feature_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_imp.head(10))

Model trained!

Top 10 Most Important Features:
             feature  importance
5              solar    0.271663
11        net_demand    0.183415
4               wind    0.135318
6               load    0.097995
2              month    0.095829
7      renewable_gen    0.051588
0               hour    0.046530
12  net_demand_ratio    0.032001
9     renewable_pct2    0.028411
8      renewable_pct    0.024612


#### Make Predictions and Create Submission

In [27]:
# Predict on test set
pred = model.predict(x_test)

# Create submission dataframe
submission = pd.DataFrame({
    'ID': x_test.index,
    'forecast': pred
})

# Save submission file
submission.to_csv('/kaggle/working/submission.csv', index=False)
print("✓ File submission.csv created")
print(f"\nFirst 10 predictions:")
print(submission.head(10))
print(f"\nPrediction stats:")
print(f"Mean: {pred.mean():.2f} €/MWh")
print(f"Std: {pred.std():.2f} €/MWh")

✓ File submission.csv created

First 10 predictions:
   ID    forecast
0   0 -133.754639
1   1 -147.378911
2   2 -133.754639
3   3 -118.790061
4   4 -150.249760
5   5 -150.249760
6   6 -133.754639
7   7 -147.378911
8   8 -147.378911
9   9 -115.919212

Prediction stats:
Mean: -11.26 €/MWh
Std: 143.74 €/MWh


### Visualise Predictions

In [28]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_fe['date'], y=pred, mode='lines', name='Predicted Spread'))
fig.update_layout(title='Predicted Spread Over Time', xaxis_title='Date', yaxis_title='Predicted Spread (€/MWh)')
fig.show()

### Cross Validation

In [ ]:
print("Running cross-validation to estimate performance...")

# TimeSeriesSplit, not KFold: each fold may only train on data preceding its
# validation window, otherwise the model scores 2020 using 2023 information.
tscv = TimeSeriesSplit(n_splits=5)
pred_cv = pd.Series(index=y_train.index, dtype=float, name='forecast')

for fold, (tr_idx, val_idx) in enumerate(tscv.split(x_train), start=1):
    fold_model = clone(model)
    fold_model.fit(x_train.iloc[tr_idx], y_train.iloc[tr_idx])
    pred_cv.iloc[val_idx] = fold_model.predict(x_train.iloc[val_idx])
    print(f"  fold {fold}/5 complete")

# Load imbalances data for PnL calculation
imbalances = pd.read_csv('/kaggle/input/ensimag-if-2025/imbalances.csv', 
                          parse_dates=['date'], index_col='date')

# Calculate PnL
def calculate_pnl(row):
    """Calculate profit/loss for a single row"""
    pos_mw = 50
    
    # Too risky - don't trade
    if abs(row['imbalances']) > 1000:
        return 0
    
    if row['forecast'] >= 0:
        # Spread >= 0 => Buy 50MW on Day-Ahead (long position)
        return pos_mw * row['spread']
    else:
        # Spread < 0 => Sell 50MW on Day-Ahead (short position)
        return -pos_mw * row['spread']

# Combine predictions with actual data
# The earliest fold is never in a validation window, so it has no out-of-fold
# prediction; without the dropna those rows fall through calculate_pnl's
# else-branch and get booked as short PnL on a NaN forecast.
concat = pd.concat([pred_cv, imbalances, y_train], axis=1).dropna(subset=['forecast'])
concat['pnl'] = concat.apply(calculate_pnl, axis=1)
concat['pnl_cum'] = concat['pnl'].cumsum()

print(f"\nEstimated Cumulated PnL: €{concat['pnl_cum'].iloc[-1]:,.2f}")

# Visualize cumulated PnL
fig = go.Figure()
fig.add_trace(go.Scatter(x=concat.index, y=concat['pnl_cum'], 
                         mode='lines', name='Cumulated PnL'))
fig.update_layout(title='Walk-Forward CV: Cumulated PnL Over Time',
                  xaxis_title='Date', yaxis_title='Cumulated PnL (€)')
fig.show()

#### Cross-validation (evaluating your model without submitting it)
You can perform cross-validation on the training dataset, for example using K-fold cross-validation. This involves learning from part of the history (80% for example) and predicting on another part (20%), then modifying the parts used several times in order to predict on the entire history. The aim is to avoid learning and predicting on the same part at the same time (this is called data leakage).

In [ ]:
''''
from sklearn.model_selection import cross_val_predict

# Change this part (model + processing on x_train)
model = LinearRegression()  # To change with your model
pred_cv = cross_val_predict(model, x_train, y_train, cv=5)


# This part compute PnL, you don't need to change code below
def calculate_pnl(row):
    pos_mw = 50
    # Too risky
    if abs(row['imbalances']) > 1000:
        return 0
    if row['forecast'] >= 0:
        # If spread >= 0 => Buy 50MW DA (pos_mw > 0) long position
        return pos_mw * row['spread']
    else:
        # If spread < 0 => Sell 50MW DA (pos_mw < 0) short position
        return -pos_mw * row['spread']


pred_cv = pd.Series(pred_cv, index=y_train.index, name='forecast')
imbalances = pd.read_csv('/kaggle/input/ensimag-if-2025/imbalances.csv', parse_dates=['date'], index_col=0)
concat = pd.concat([pred_cv, imbalances, y_train], axis=1)
concat['pnl'] = concat.apply(calculate_pnl, axis=1)
concat['pnl_cum'] = concat['pnl'].cumsum()
print(f"Cumulated PnL: {concat['pnl_cum'].iloc[-1]}€")


# Show cumulated PnL
fig = go.Figure()
fig.add_trace(go.Scatter(x=concat.index, y=concat['pnl_cum'], mode='lines', name='Cumulated PnL'))
fig.show()
''''''